<a href="https://colab.research.google.com/github/tripti369/AI-voice-detector-/blob/main/AI___voice___Detector_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install librosa numpy pandas scikit-learn resampy

import os
import zipfile
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle

In [ ]:
from google.colab import files

# Upload the audio zip file
uploaded = files.upload()

Saving Audio file -20260201T134527Z-3-001.zip to Audio file -20260201T134527Z-3-001 (3).zip


In [ ]:
import os
import subprocess
import sys
try:
    import resampy
    print("✅ resampy is already installed.")
except ImportError:
    print("⚠️ resampy not found. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "resampy"])
    print("✅ resampy installed successfully!")

import zipfile
import librosa
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
if 'uploaded' not in locals() or len(uploaded) == 0:
    print("❌ Error: 'uploaded' variable is empty.")
    print("Please run the 'uploaded = files.upload()' cell again.")
else:
    zip_filename = next(iter(uploaded))
    zip_path = f'/content/{zip_filename}'
    extract_path = '/content/dataset'

    print(f"✅ Found file: {zip_filename}")

    if not os.path.exists(extract_path):
        os.makedirs(extract_path)
        print(f"Extracting {zip_filename}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Extraction Complete.")
    else:
        print("Folder already exists, skipping extraction.")
    ai_files = []
    human_files = []

    print("\nScanning extracted folders for audio...")
    for root, dirs, files in os.walk(extract_path):
        is_ai = 'AI' in root or 'AI_Audio' in root
        is_human = 'Human' in root or 'Human_Audio' in root or 'Manual' in root
        if 'ai_audio_filenames' in locals() and 'audio_dir' in locals():
            pass
    features = []
    labels = []

    def extract_features(file_path):
        try:
            # Removed 'kaiser_fast' to reduce dependency errors if resampy fails again
            audio, sr = librosa.load(file_path, sr=22050)
            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            return np.mean(mfccs.T, axis=0)
        except Exception as e:
            # Verify if file actually exists before complaining
            if not os.path.exists(file_path):
                print(f"⚠️ Skipped missing file: {os.path.basename(file_path)}")
            else:
                print(f"⚠️ Error reading {os.path.basename(file_path)}: {e}")
            return None

    # Logic to handle your specific previous list setup if it exists
    if 'ai_audio_filenames' in locals() and 'audio_dir' in locals():
        print("\nUsing your manually categorized lists...")

        # AI FILES
        print(f"Processing {len(ai_audio_filenames)} AI files...")
        for f_name in ai_audio_filenames:
            full_path = os.path.join(audio_dir, f_name)
            data = extract_features(full_path)
            if data is not None:
                features.append(data)
                labels.append(1)

        # HUMAN FILES
        print(f"Processing {len(human_audio_filenames)} Human files...")
        for f_name in human_audio_filenames:
            full_path = os.path.join(audio_dir, f_name)
            data = extract_features(full_path)
            if data is not None:
                features.append(data)
                labels.append(0)

    else:
        # FALLBACK: If manual lists are missing, search automatically
        print("\nManual lists not found. Searching folders automatically...")
        for root, dirs, files in os.walk(extract_path):
            for f in files:
                if f.endswith('.mp3'):
                    path = os.path.join(root, f)
                    # Guess label based on folder name
                    if 'AI' in root:
                        data = extract_features(path)
                        if data: features.append(data); labels.append(1)
                    elif 'Human' in root or 'Original' in root:
                        data = extract_features(path)
                        if data: features.append(data); labels.append(0)

    if len(features) > 0:
        X = np.array(features)
        y = np.array(labels)

        print(f"\n✅ Successfully processed {len(X)} files.")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        model = RandomForestClassifier(n_estimators=100)
        model.fit(X_train, y_train)

        acc = accuracy_score(y_test, model.predict(X_test))
        print(f"\n🎉 TRAINING SUCCESS! Accuracy: {acc*100:.2f}%")

        with open('voice_detection_model.pkl', 'wb') as f:
            pickle.dump(model, f)
        print("Model saved as 'voice_detection_model.pkl'")
    else:
        print("\n❌ Error: No features extracted. Check paths or 'resampy' install.")

✅ resampy is already installed.
✅ Found file: Audio file -20260201T134527Z-3-001 (3).zip
Folder already exists, skipping extraction.

Scanning extracted folders for audio...

Using your manually categorized lists...
Processing 11 AI files...
Processing 39 Human files...
⚠️ Skipped missing file: ఈ దేశത്തെ ప్రధానమన.mp3


/tmp/ipython-input-2455423464.py:52: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=22050)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


⚠️ Skipped missing file: ఇదెല്ലാം నాకు ఒక.mp3
⚠️ Skipped missing file: പൂണ സോമ്പേറി നായകൻ.mp3
⚠️ Skipped missing file: നക്ഷത്രങ്ങൾ എൻപവാ.mp3
⚠️ Skipped missing file: ശ്രുതി സ്കൂൾ കി രാദ.mp3
⚠️ Skipped missing file: ഈరోజు കാലാവസ്ഥ നല്ലതല്ല.mp3
⚠️ Skipped missing file: ഞാൻ അമ്മയോടൊപ്പം ക്ഷേത്രത്തിൽ പോകുന്നു.mp3
⚠️ Skipped missing file: വിജയങ്ങൾ മനോഹരമായിരിക്കണം.mp3

✅ Successfully processed 42 files.

🎉 TRAINING SUCCESS! Accuracy: 77.78%
Model saved as 'voice_detection_model.pkl'


In [ ]:
import pickle
import librosa
import numpy as np
from google.colab import files
import os

# Load the trained model
with open('voice_detection_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
print("✅ Model 'voice_detection_model.pkl' loaded successfully.")

✅ Model 'voice_detection_model.pkl' loaded successfully.


Next, upload an audio file you want to classify (AI or Human). This can be a new file not used in training.

In [ ]:
# Upload the audio file for testing
uploaded_test = files.upload()

if len(uploaded_test) == 0:
    print("❌ Error: No file uploaded. Please upload an audio file to test.")
else:
    test_audio_filename = next(iter(uploaded_test))
    test_audio_path = f'/content/{test_audio_filename}'
    print(f"✅ Test file: {test_audio_filename} uploaded.")

    # Feature extraction function (re-using the one from training)
    def extract_features_for_prediction(file_path):
        try:
            import resampy # Ensure resampy is imported here too
            audio, sr = librosa.load(file_path, res_type='kaiser_fast')
            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            return np.mean(mfccs.T, axis=0)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            return None

    # Extract features from the uploaded test audio
    test_features = extract_features_for_prediction(test_audio_path)

    if test_features is not None:
        # Reshape the features to be a 2D array (1 sample, n_features)
        test_features = test_features.reshape(1, -1)

        # Make prediction
        prediction = loaded_model.predict(test_features)

        # Interpret prediction
        if prediction[0] == 1:
            print(f"\n🤖 Prediction for '{test_audio_filename}': This audio is likely **AI-generated**.")
        else:
            print(f"\n👤 Prediction for '{test_audio_filename}': This audio is likely **Human-recorded**.")
    else:
        print("❌ Could not extract features from the test audio file.")

Saving Audio file -20260201T134527Z-3-001.zip to Audio file -20260201T134527Z-3-001 (4).zip
✅ Test file: Audio file -20260201T134527Z-3-001 (4).zip uploaded.


/tmp/ipython-input-3847407546.py:15: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, res_type='kaiser_fast')
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Error processing /content/Audio file -20260201T134527Z-3-001 (4).zip: 
❌ Could not extract features from the test audio file.


In [ ]:
import resampy # Moved to top-level to resolve librosa lazy loading issues

test_audio_filename_from_dataset = 'ENG_US_F_MelissaM.mp3' # You can change this to any other audio file name from your dataset
test_audio_path_from_dataset = os.path.join(audio_dir, test_audio_filename_from_dataset)

print(f"✅ Testing with file from dataset: {test_audio_path_from_dataset}")

# Feature extraction function (re-using the one from training)
def extract_features_for_prediction(file_path):
    try:
        # Removed 'res_type='kaiser_fast'' and set sr=22050 to match training
        audio, sr = librosa.load(file_path, sr=22050)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        return np.mean(mfccs.T, axis=0)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Extract features from the uploaded test audio
test_features = extract_features_for_prediction(test_audio_path_from_dataset)

if test_features is not None:
    # Reshape the features to be a 2D array (1 sample, n_features)
    test_features = test_features.reshape(1, -1)

    # Make prediction
    prediction = loaded_model.predict(test_features)

    # Interpret prediction
    if prediction[0] == 1:
        print(f"\n🤖 Prediction for '{test_audio_filename_from_dataset}': This audio is likely **AI-generated**.")
    else:
        print(f"\n👤 Prediction for '{test_audio_filename_from_dataset}': This audio is likely **Human-recorded**.")
else:
    print("❌ Could not extract features from the test audio file.")

✅ Testing with file from dataset: /content/dataset/Audio file /ENG_US_F_MelissaM.mp3

👤 Prediction for 'ENG_US_F_MelissaM.mp3': This audio is likely **Human-recorded**.
